# set up

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import rclpy
from rclpy.node import Node
from geometry_msgs.msg import Pose
from sensor_msgs.msg import JointState

from pymoveit2 import MoveIt2
from tf2_ros import Buffer, TransformListener

import json
import time

import numpy as np
import pickle

## node

In [3]:
rclpy.init()

node = Node("demo_node3")

print("ROS initialized")

ROS initialized


In [4]:
from rclpy.logging import LoggingSeverity
node.get_logger().set_level(LoggingSeverity.ERROR)

# Arm initialize

## moveit

In [5]:
def spin_some(duration=2.0):
    """    
    important: take some time to process incoming data!
    """
    import time
    end = time.time() + duration
    while time.time() < end:
        rclpy.spin_once(node, timeout_sec=0.1)

In [6]:
from pymoveit2 import MoveIt2
from tf2_ros import Buffer, TransformListener

from geometry_msgs.msg import Pose

tf_buffer = Buffer()
tf_listener = TransformListener(tf_buffer, node)

spin_some() # give it some time to update

In [7]:
from perception_interfaces.srv import PerceptionCheck

def call_perception(node, path, mode):
    client = node.create_client(PerceptionCheck, 'perception_check')

    while not client.wait_for_service(timeout_sec=1.0):
        print("Waiting for perception service...")

    req = PerceptionCheck.Request()
    req.file_path = path
    req.mode = mode

    future = client.call_async(req)
    rclpy.spin_until_future_complete(node, future)

    return future.result().result

## arm and gripper

In [8]:
joint_state = None

def joint_callback(msg):
    global joint_state
    joint_state = msg

node.create_subscription(
    JointState,
    "/joint_states",
    joint_callback,
    10
)

spin_some(1)

print(joint_state)

sensor_msgs.msg.JointState(header=std_msgs.msg.Header(stamp=builtin_interfaces.msg.Time(sec=1778275813, nanosec=176335573), frame_id='base_link'), name=['right_finger_bottom_joint', 'joint_1', 'joint_2', 'joint_4', 'joint_5', 'joint_3', 'joint_6'], position=[-0.004931605356931686, 0.2523925701967715, -1.1750644372467898, 0.31469322915754017, 1.4199639374005149, 1.3741996445963751, 0.9881954835620012], velocity=[0.0, 2.383047225387026e-05, -3.700707292170152e-05, 3.702765536634454e-05, 0.0, 0.0, -3.436336864207654e-05], effort=[nan, -0.3230360150337219, -8.451531410217285, 0.4958707094192505, -0.9052136540412903, 1.2577784061431885, 0.15641559660434723])


In [9]:
moveit2 = MoveIt2(
    node=node,
    joint_names=[
        "joint_1","joint_2","joint_3",
        "joint_4","joint_5","joint_6",
    ],
    base_link_name="base_link",
    end_effector_name="tool_frame",
    group_name="arm",
)

In [10]:
from rclpy.action import ActionClient
from control_msgs.action import GripperCommand

gripper_client = ActionClient(
    node,
    GripperCommand,
    "/gen3_lite_2f_gripper_controller/gripper_cmd"
)


def control_gripper(position, effort=1.0):
    """
    position:
        0.0 = open
        ~0.8–1.0 = closed
    """

    goal_msg = GripperCommand.Goal()
    goal_msg.command.position = position
    goal_msg.command.max_effort = effort

    # print(f"Sending gripper command: {position}")

    # wait for server
    gripper_client.wait_for_server()

    send_goal_future = gripper_client.send_goal_async(goal_msg)

    rclpy.spin_until_future_complete(node, send_goal_future)
    goal_handle = send_goal_future.result()

    if not goal_handle.accepted:
        print("Gripper goal rejected")
        return

    result_future = goal_handle.get_result_async()
    rclpy.spin_until_future_complete(node, result_future)

    # print("Gripper done")

def open_gripper():
    control_gripper(0.0)

def close_gripper():
    control_gripper(0.8)  # adjust if needed

In [11]:
close_gripper()
open_gripper()

# Pose

## functions

### joint angle

In [12]:
def get_current_joint_positions(target_joint_names=None):
    global joint_state

    while joint_state is None:
        rclpy.spin_once(node, timeout_sec=0.1)

    name_to_pos = dict(zip(joint_state.name, joint_state.position))

    if target_joint_names is None:
        target_joint_names = [
            "joint_1",
            "joint_2",
            "joint_3",
            "joint_4",
            "joint_5",
            "joint_6",
        ]

    return [name_to_pos[name] for name in target_joint_names]

    

def reached_joint_pose(target, current=None, tol=0.08):
    if current is None:
        current = get_current_joint_positions()
        
    errors = [abs(t - c) for t, c in zip(target, current)]

    return max(errors) < tol

### poses

In [13]:
def get_pose():
    spin_some(1.0)
    try:
        trans = tf_buffer.lookup_transform(
            "base_link",
            "tool_frame",
            rclpy.time.Time()
        )

        pose = Pose()
        pose.position.x = trans.transform.translation.x
        pose.position.y = trans.transform.translation.y
        pose.position.z = trans.transform.translation.z
        pose.orientation = trans.transform.rotation

        return pose
    except Exception as e:
        print("TF error:", e)
        return None
    
def print_pose(pose):
    print("translation:")
    print([pose.position.x, pose.position.y, pose.position.z])
    print("rotation")
    print([pose.orientation.x,
           pose.orientation.y,
           pose.orientation.z,
           pose.orientation.w])

test_pose0 = get_pose()
if test_pose0:
    print("Current pose:")
    print_pose(test_pose0)
else:
    print("failed to get pose.")
    

def make_pose(pos_xyz, orient_xyzw, pose=None):
    pose = Pose()
    
    pose.position.x, pose.position.y, pose.position.z = pos_xyz
    pose.orientation.x, pose.orientation.y, pose.orientation.z, pose.orientation.w = orient_xyzw
    
    return pose

Current pose:
translation:
[0.353699523691948, -0.1290431706160597, 0.04820508706586096]
rotation
[0.8043922346119823, 0.02698821066418206, 0.023821928188577805, 0.5930069857248446]


In [14]:
DEFAULT_SPEED = 0.2


def sync_state():
    time.sleep(0.5)          # let controller settle
    spin_some(0.5)           # update TF + joints

def reached_pose(target, current=None, pos_tol=0.01, 
                 ori_tol=0.05, check_orient=False):
    
    if current is None:
        current = get_pose()
        
    # position
    p1 = np.array([
        current.position.x,
        current.position.y,
        current.position.z
    ])

    p2 = np.array([
        target.position.x,
        target.position.y,
        target.position.z
    ])

    pos_error = np.linalg.norm(p1 - p2)

    if not check_orient:
        return pos_error < pos_tol

    else:
        # orientation (quaternion distance)
        q1 = np.array([
            current.orientation.x,
            current.orientation.y,
            current.orientation.z,
            current.orientation.w
        ])

        q2 = np.array([
            target.orientation.x,
            target.orientation.y,
            target.orientation.z,
            target.orientation.w
        ])

        ori_error = np.linalg.norm(q1 - q2)

        return pos_error < pos_tol and ori_error < ori_tol


### move to pose

In [15]:

def move_to_pose_robust(p, speed=None):
    sync_state()

    if speed is None:
        speed = DEFAULT_SPEED

    moveit2.max_velocity = speed
    moveit2.max_acceleration = speed

    plan = moveit2.plan(pose=p)

    if plan is None:
        print("❌ Planning failed")
        return False

    moveit2.execute(plan)
    moveit2.wait_until_executed()

    ok = reached_pose(target=p, current=get_pose())

    if not ok:
        print("⚠️ Arm did not reach target pose")
        return False
    else:
        print("✅ Armd reached target pose")
        return True

In [16]:
# move_to_pose_robust(grey_bottle_grasp_pose)
# close_gripper()
# move_to_pose_robust(cup_pen_grasp_pose)
# open_gripper()

### move by plan

In [17]:

def get_start_and_end_joint_targets(plan):
    joint_names = list(plan.joint_names)
    start_positions = list(plan.points[0].positions)
    end_positions = list(plan.points[-1].positions)

    return joint_names, start_positions, end_positions

def move_by_plan(moveit_plan_pkl_path: str, timeout=10.0, tol=0.03):
    sync_state()

    with open(moveit_plan_pkl_path, "rb") as f:
        plan = pickle.load(f)

    joint_names, start_target, end_target = get_start_and_end_joint_targets(plan)
    
    # if not reached_joint_pose(target=start_target):
    # this part is doing something useful to some trajectories
    # make it unconditional
    # print("Moving arm to trajectory start...")
    moveit2.move_to_configuration(
        joint_positions=start_target,
        joint_names=joint_names
        )
    moveit2.wait_until_executed()

    print("Arm is in starting position. Executing saved trajectory...")

    sync_state()
    moveit2.execute(plan)
    moveit2.wait_until_executed()
    
    ok = reached_joint_pose(target=end_target)
    if not ok:
        print("⚠️ Arm did not reach endpoint")
        return False
    else:
        print("✅ Trajectory completed")
        return True

In [18]:
def inspect_plan_start(plan):
    traj = plan.joint_trajectory if hasattr(plan, "joint_trajectory") else plan

    print("Joint names:", traj.joint_names)
    print("Number of points:", len(traj.points))

    for i in range(min(5, len(traj.points))):
        p = traj.points[i]
        t = p.time_from_start.sec + p.time_from_start.nanosec * 1e-9
        print(f"\nPoint {i}, t={t:.4f}")
        print("pos:", list(p.positions))
        print("vel:", list(p.velocities))
        print("acc:", list(p.accelerations))

    if len(traj.points) >= 2:
        p0 = traj.points[0]
        p1 = traj.points[1]

        t0 = p0.time_from_start.sec + p0.time_from_start.nanosec * 1e-9
        t1 = p1.time_from_start.sec + p1.time_from_start.nanosec * 1e-9
        dt = t1 - t0

        dq = [abs(a - b) for a, b in zip(p0.positions, p1.positions)]

        print("\nFirst segment:")
        print("dt:", dt)
        print("dq:", dq)
        print("approx velocity:", [x / dt for x in dq])

In [19]:
BOTTLE_TO_SHAKE_TRAJ = "demo/config/trajectory_bottle2shake.pkl"
SHAKE_TO_BOTTLE_TRAJ = "demo/config/trajectory_shake2bottle.pkl"

CUP_TO_TILT_TRAJ = "demo/config/trajectory_cup_grasp2tilt.pkl"
TILT_TO_CUP_TRAJ = "demo/config/trajectory_tilt2cup_grasp.pkl"


In [20]:
# # move_to_pose_robust(grey_bottle_grasp_pose)
# move_by_plan(CUP_TO_TILT_TRAJ)
# move_by_plan(TILT_TO_CUP_TRAJ)
# move_by_plan(BOTTLE_TO_SHAKE_TRAJ)
# move_by_plan(SHAKE_TO_BOTTLE_TRAJ)

## set pose

In [21]:
drop_off_pose = get_pose()
print_pose(drop_off_pose)

drop_off_pose = make_pose(
    pos_xyz=
[0.3590171175249066, 0.5474951181499795, 0.09054964349581472]
    ,
orient_xyzw =
[-0.007259383560778488, 0.727832403750572, 0.6848156934362338, 0.03513914376350253]
)

translation:
[0.35369521903912576, -0.12904353327630969, 0.04820698915000288]
rotation
[0.8043879255905704, 0.026993595448914803, 0.023816529202034375, 0.5930128024803579]


### bottle

In [22]:
grey_bottle_grasp_pose = make_pose(
    pos_xyz=(
        0.5037365889870258, 
        -0.2240841929310071, 
        0.11609242244597344
        ), 
    orient_xyzw=(
        0.7301995308282448, 
        -0.005421623908711407, 
        0.033587134403944244, 
        0.6823863682511068
        )
    )

grey_bottle_pre_grasp_pose =  make_pose(
    pos_xyz=[0.5035170876022704, -0.12519245872336004, 0.13016054581559786],
    orient_xyzw=[0.7122703498240701, 0.018009366124129053, 
                 0.044352911487721705, 0.700270969508137]
)

above_mircophone_pose = make_pose(
    pos_xyz=[
        0.5035170876022704,
        -0.12519245872336004,
        0.13016054581559786 + 0.20
    ],
    orient_xyzw=[
        0.7122703498240701,
        0.018009366124129053,
        0.044352911487721705,
        0.700270969508137
    ]
)

### cup

In [23]:
cup_pen_pre_grasp_pose = make_pose(
    pos_xyz=[0.35346299290800076, -0.12890278627088447, 0.04871254200241307],
    orient_xyzw=[0.8039581741893985, 0.026671357129026064, 0.024053949195080093, 0.5936002867174723]
)

cup_pen_grasp_pose = make_pose(
    pos_xyz=[0.3470795335833769, -0.17505817482321506, 0.048968048074364764],
    orient_xyzw=[0.7999772075325435, -0.0059638605729074095, 0.00583137462724921, 0.5999724117536216]
)

cup_pre_tilt_pose = make_pose(
    pos_xyz=[0.34769987911092504, -0.1751015310279324, 0.24681058864261823],
    orient_xyzw=[0.8005613986661017, -0.005208209991114569, 0.00414769912108677, 0.5992137499310779]
)

cup_tilt_pose = make_pose(
    pos_xyz=[0.28762861267741296, -0.1889354706791522, 0.20975108669772796],
    orient_xyzw=[0.9729684284898917, 0.1697256380619787, 0.03949216785938783, -0.1515454176942695]
)

## test poses

In [24]:
# move_to_pose_robust(cup_pen_grasp_pose)
# move_to_pose_robust(grey_bottle_grasp_pose)


# sensers

## microphone

In [25]:
import subprocess
import os
from datetime import datetime

audio_proc = None
current_audio_file = None

def start_audio_record(trial_num, save_path="demo/audio"):
    global audio_proc, current_audio_file

    os.makedirs(save_path, exist_ok=True)

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    filename = f"trial_{trial_num}_{timestamp}.wav"
    full_path = os.path.join(save_path, filename)

    cmd = [
        "arecord",
        "-D", "plughw:2,0",
        "-f", "S16_LE",
        "-r", "16000",
        "-c", "1",
        full_path
    ]

    print(f"🎤 Start recording → {full_path}")

    audio_proc = subprocess.Popen(cmd)
    current_audio_file = full_path

    return full_path

def stop_audio_record():
    global audio_proc, current_audio_file

    if audio_proc is not None:
        print("⏹ Stop recording")

        audio_proc.terminate()
        audio_proc.wait()

        audio_proc = None

        # 🔥 VERY IMPORTANT: let file finalize
        time.sleep(0.2)

        return current_audio_file

    return None

# start_audio_record(trial_num=0)
# time.sleep(2.0)
# stop_audio_record()

In [26]:
laptop_proc = None
laptop_file = None

def start_laptop_record(trial_num, save_path="demo/audio"):

    global laptop_proc, laptop_file

    os.makedirs(save_path, exist_ok=True)

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

    filename = f"laptop_{trial_num}_{timestamp}.wav"

    full_path = os.path.join(save_path, filename)

    cmd = [
        "arecord",
        "-D", "plughw:1,0",
        "-f", "S16_LE",
        "-r", "16000",
        "-c", "2",   # IMPORTANT: laptop mic needs stereo
        full_path
    ]

    print(f"💻 Laptop mic recording → {full_path}")

    laptop_proc = subprocess.Popen(cmd)

    laptop_file = full_path

    return full_path


def stop_laptop_record():

    global laptop_proc

    if laptop_proc is not None:

        print("⏹ Stop laptop recording")

        laptop_proc.terminate()
        laptop_proc.wait()

        laptop_proc = None

        time.sleep(0.2)

    return laptop_file

In [27]:
start_laptop_record(0)
start_audio_record(0)
time.sleep(1)
stop_laptop_record()
stop_audio_record()

💻 Laptop mic recording → demo/audio/laptop_0_20260508_173016.wav
🎤 Start recording → demo/audio/trial_0_20260508_173016.wav


Recording WAVE 'demo/audio/trial_0_20260508_173016.wav' : Signed 16 bit Little Endian, Rate 16000 Hz, Mono
Recording WAVE 'demo/audio/laptop_0_20260508_173016.wav' : Signed 16 bit Little Endian, Rate 16000 Hz, Stereo


⏹ Stop laptop recording


Aborted by signal Terminated...
arecord: pcm_read:2221: read error: Interrupted system call


⏹ Stop recording


Aborted by signal Terminated...
arecord: pcm_read:2221: read error: Interrupted system call


'demo/audio/trial_0_20260508_173016.wav'

## camera

In [28]:
import cv2
import os
from datetime import datetime
import time

def capture_image(trial_num, save_path="./image"):
    global capture_flag, latest_image

    os.makedirs(save_path, exist_ok=True)

    capture_flag = True

    sync_state()  # important!

    # wait for image
    timeout = 2.0
    t0 = time.time()

    while capture_flag and (time.time() - t0 < timeout):
        time.sleep(0.05)

    if latest_image is None:
        print("⚠️ No image captured")
        return None

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    filename = f"trial_{trial_num}_{timestamp}.png"
    full_path = os.path.join(save_path, filename)

    cv2.imwrite(full_path, latest_image)

    print(f"📸 Saved image: {full_path}")

    return full_path


In [29]:
from cv_bridge import CvBridge

latest_image = None
capture_flag = False

recording_video = False
video_frames = []


bridge = CvBridge()

def image_callback(msg):
    global latest_image, capture_flag
    global recording_video, video_frames

    frame = bridge.imgmsg_to_cv2(msg, desired_encoding="bgr8")
    latest_image = frame.copy()

    if capture_flag:
        capture_flag = False

    if recording_video:
        video_frames.append(frame.copy())

In [30]:
video_frames = []
recording_video = False


def start_video_recording():
    global video_frames, recording_video
    video_frames = []
    recording_video = True
    print("🎥 Recording started")


def stop_video_recording(name="video", save_path="./video", fps=15):
    global video_frames, recording_video

    recording_video = False

    if len(video_frames) == 0:
        print("❌ No frames recorded")
        return None

    os.makedirs(save_path, exist_ok=True)

    h, w = video_frames[0].shape[:2]
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    path = os.path.join(save_path, f"{name}_{timestamp}.avi")

    fourcc = cv2.VideoWriter_fourcc(*"MJPG")
    writer = cv2.VideoWriter(path, fourcc, fps, (w, h))

    if not writer.isOpened():
        print("❌ VideoWriter failed to open")
        return None

    for frame in video_frames:
        if frame.shape[:2] == (h, w):
            writer.write(frame)

    writer.release()

    print(f"✅ Saved {len(video_frames)} frames to {path}")
    return path

In [31]:
from sensor_msgs.msg import Image as ROSImage
from IPython.display import Image as DisplayImage

image_sub = node.create_subscription(
    ROSImage,
    "/camera/camera/color/image_raw",
    image_callback,
    10
)
capture_image(trial_num="start_sub")

📸 Saved image: ./image/trial_start_sub_20260508_173019.png


'./image/trial_start_sub_20260508_173019.png'

In [32]:
def record_for_seconds(seconds=5.0, name="test", save_path="demo/video", fps=15):
    start_video_recording()

    t0 = time.time()
    while time.time() - t0 < seconds:
        rclpy.spin_once(node, timeout_sec=0.05)

    return stop_video_recording(name, save_path, fps=fps)

In [33]:
# record_for_seconds(1)

In [34]:
# start_video_recording()
# capture_image(trial_num="test_test")
# stop_video_recording("Test", "demo/video", fps=30)

# behaviors

## motions

In [35]:
import numpy as np
import time

ARM_JOINT_NAMES = [
    "joint_1",
    "joint_2",
    "joint_3",
    "joint_4",
    "joint_5",
    "joint_6",
]


def lift(delta_z=0.30):
    pose = get_pose()
    pose.position.z += delta_z
    move_to_pose_robust(pose)

def get_arm_joints():
    joint_dict = dict(zip(joint_state.name, joint_state.position))
    return [joint_dict[name] for name in ARM_JOINT_NAMES]

def reached_joint(target_joints, tol=0.05):
    current = get_arm_joints()
    return np.linalg.norm(np.array(current) - np.array(target_joints)) < tol

def sync_state(wait=0.3):
    time.sleep(wait)
    spin_some(wait)

def rotate_wrist(delta, speed=0.5, max_trial=3):
    joint_idx = moveit2.joint_names.index("joint_6")

    # --- sync once ---
    sync_state()

    joints = get_arm_joints()
    target = joints.copy()
    target[joint_idx] += delta

    # --- save speed ---
    prev_vel = moveit2.max_velocity
    prev_acc = moveit2.max_acceleration

    moveit2.max_velocity = speed
    moveit2.max_acceleration = speed

    # --- plan ONCE ---
    plan = moveit2.plan(joint_positions=target)

    if plan is None:
        print("❌ Planning failed")
        return False

    # --- execute multiple times ---
    for trial in range(max_trial):
        print(f"Execute trial {trial+1}")

        moveit2.execute(plan)
        moveit2.wait_until_executed()
        
        # --- relaxed verification ---
        if reached_joint(target, tol=0.05):   # looser tolerance
            print("✅ Success")
            break

    else:
        print("⚠️ Not perfectly reached, but continuing")

    # --- restore speed ---
    moveit2.max_velocity = prev_vel
    moveit2.max_acceleration = prev_acc

    return True


## shake

In [36]:
def shake_bottle(test_num, audio_save_path="demo/audio"):
    print("=== Shake Bottle Behavior ===")

    # move to shake position
    ready_to_shake = False
    tolerance = 0
    while not ready_to_shake and tolerance < 3:
        ready_to_shake = move_by_plan(
            moveit_plan_pkl_path=BOTTLE_TO_SHAKE_TRAJ,
        )
        tolerance += 1
    if not ready_to_shake:
        print(f"can't start shaking behavior.")
        return False

    print("start recording audio...")
    start_audio_record(
        trial_num=f"test{test_num}",
        save_path=audio_save_path
    )

    time.sleep(0.3)  # mic warmup

    rotate_wrist(1.57 * 2, speed=2.0)

    time.sleep(0.3)  # capture tail
    audio_file = stop_audio_record()

    audio_file = os.path.abspath(audio_file)

    print(f"[Audio] Recorded file: {audio_file}")

    # CALL PERCEPTION SERVICE
    with_water = call_perception(node, audio_file, "audio")

    if with_water == 1:
        print("✅ Water detected")
    elif with_water == 0:
        print("❌ Empty bottle")
    else:
        print("⚠️ Audio error")

    # print("returning to bottle position...")
    # move_to_pose_robust(above_mircophone_pose)
    rotate_wrist(-1.57 * 2, speed=2.0)

    if with_water == 1:
        return True
    elif with_water == 0:
        return False
    else:
        return None

# example usage
# with_water = shake_bottle(
#     test_num=0,
#     audio_save_path="demo/audio"
# )

In [37]:
# with_water = shake_bottle(
#     test_num=0,
#     audio_save_path="demo/audio"
# )

## tilt

In [38]:
import matplotlib
matplotlib.use("Agg")

import matplotlib.pyplot as plt
from perception_module.utils import crop_cup_region, detect_pen
from IPython.display import Image, display


def visualize_detection_image(img_file, save_dir="demo/vis"):
    os.makedirs(save_dir, exist_ok=True)

    img = cv2.imread(img_file)
    if img is None:
        print(f"❌ Failed to read image: {img_file}")
        return None

    cropped = crop_cup_region(img)
    hsv = cv2.cvtColor(cropped, cv2.COLOR_BGR2HSV)

    lower_blue = np.array([100, 100, 50])
    upper_blue = np.array([130, 255, 255])
    lower_orange = np.array([5, 100, 100])
    upper_orange = np.array([20, 255, 255])

    mask_blue = cv2.inRange(hsv, lower_blue, upper_blue)
    mask_orange = cv2.inRange(hsv, lower_orange, upper_orange)
    mask = mask_blue + mask_orange

    kernel = np.ones((3, 3), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
    # mask = cv2.inRange(hsv, lower_orange, upper_orange)

    contours, _ = cv2.findContours(
        mask,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    vis = cropped.copy()
    cv2.drawContours(vis, contours, -1, (0, 255, 0), 2)

    areas = [cv2.contourArea(c) for c in contours]
    max_area = max(areas) if areas else 0

    fig = plt.figure(figsize=(12, 3))

    plt.subplot(1, 4, 1)
    plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.title("Captured")
    plt.axis("off")

    plt.subplot(1, 4, 2)
    plt.imshow(cv2.cvtColor(cropped, cv2.COLOR_BGR2RGB))
    plt.title("Cropped")
    plt.axis("off")

    plt.subplot(1, 4, 3)
    plt.imshow(mask, cmap="gray")
    plt.title("Mask")
    plt.axis("off")

    plt.subplot(1, 4, 4)
    plt.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
    plt.title(f"Contours\nMax area: {max_area:.1f}")
    plt.axis("off")

    plt.suptitle(os.path.basename(img_file))
    plt.tight_layout()

    out_file = os.path.join(
        save_dir,
        f"vis_{os.path.basename(img_file)}"
    )

    plt.savefig(out_file, dpi=150)
    plt.close(fig)

    print(f"[Vision] Saved visualization: {out_file}")

    display(Image(out_file))
    return out_file

In [39]:
def tilt_cup(test_num, image_path="demo/image"):
    
    print("=== Tilt Cup Behavior ===")

    print("moving to camera-ready tilted pose...")
    move_by_plan(CUP_TO_TILT_TRAJ)

    time.sleep(.5)  # camera buffer

    print("detecting pens...")
    img_file = capture_image(
        trial_num=f"test{test_num}",
        save_path=image_path
    )

    img_file = os.path.abspath(img_file)
    visualize_detection_image(img_file)

    # print(f"[Vision] Captured image: {img_file}")

    # 🔥 CALL PERCEPTION SERVICE
    result = call_perception(node, img_file, "vision")

    if result == 1:
        print("✅ Pen detected")
    elif result == 0:
        print("❌ No pen detected")
    else:
        print("⚠️ Vision error")

    if result == 1:
        return True
    elif result == 0:
        return False
    else:
        return None

In [40]:
# open_gripper()
# move_to_pose_robust(cup_pen_grasp_pose)
# # place the cup

In [41]:
# close_gripper()
# with_pen = tilt_cup(
#     test_num=0,
#     image_path="demo/image"
# )
# with_pen

# LLM

In [42]:
import json
import requests

# OLLAMA_MODEL = "qwen2.5:0.5b"   # or whatever appears in `ollama list`
# OLLAMA_MODEL = "llama3.2:1b"
OLLAMA_MODEL = "llama3:latest"


def ask_llm(prompt, num_predict=100):
    print(f"Wating for LLM's response ...")
    response = requests.post(
        "http://localhost:11434/api/generate",
        json={
            "model": OLLAMA_MODEL,
            "prompt": prompt,
            "stream": False,
            "options": {
                "num_predict": num_predict,
                "temperature": 0
            }
        }
    )
    data = response.json()
    if "error" in data:
        raise RuntimeError(data["error"])
    return data["response"].strip()

def parse_command_fallback(command):
    c = command.lower()

    if "bottle" in c:
        return {
            "object": "bottle",
            "property": "water" if "water" in c else "empty",
            "action": "shake and listen"
        }

    elif "cup" in c:
        return {
            "object": "cup",
            "property": "pen" if "pen" in c else "empty",
            "action": "tilt and look"
        }

    return {
        "object": "unknown",
        "property": "unknown",
        "action": "ask again"
    }


def parse_command_llm_with_reasons(command, num_predict=100):
    print(f"command: {command}")
    prompt = f"""
        You are in a robotic system given command to fetch the object in this command. To do so, the robot havs to interact with available containers until it finds the one with the property required.
        Your job is to 1. parse the command to identify the container asked for. 2. decide ONE sensing action to check this container's property.

        Available sensing actions:
        - shake and listen: rotate the container upside down to give sound evidence about internal moving contents.
        - tilt and look: tilt container's opening towards the camera to give visual evidence inside an open container.

        Containers to interact with: 
        - bottles with water, sealed and non-transparent
        - bottles without water, sealed and non-transparent 
        - cups with pens, open.
        - cups without pens, open.
        
        Sensors: robot has a microphone to hear and camera to see.

        Command: "{command}". 

        Your tasks:
        1. Identify the retuired object in the command.
        2. Identify the required property.
        3. Identify the sensing action in this command.

        Return ONLY a valid JSON string.
        Do not include markdown.
        Do not include extra text outside the JSON.

        JSON format:
        {{
        "object": "bottle" or "cup",
        "property": "water" or "empty" or "pen",
        "action": "shake and listen" or "tilt and look",
        "reason": your concise reasons (2-3 bulletpoints) for choosing this action for this propety of this object,
        }}
        """
        
    
    plan = ask_llm(prompt, num_predict=num_predict)
    return plan

def parse_command_llm(command, num_predict=40):
    print(f"command: {command}")
    prompt = f"""
        You are in a robotic system given command to fetch the object in this command. To do so, the robot havs to interact with available containers until it finds the one with the property required.
        Your job is to 1. parse the command to identify the container asked for. 2. decide ONE sensing action to check this container's property.

        Available sensing actions:
        - shake and listen: rotate the container upside down to give sound evidence about internal moving contents.
        - tilt and look: tilt container's opening towards the camera to give visual evidence inside an open container.

        Containers to interact with: 
        - bottles with water, sealed and non-transparent
        - bottles without water, sealed and non-transparent 
        - cups with pens, open.
        - cups without pens, open.
        
        Sensors: robot has a microphone to hear and camera to see.

        Command: "{command}". 

        Your tasks:
        1. Identify the retuired object in the command.
        2. Identify the required property.
        3. Identify the sensing action in this command.

        Return ONLY a valid JSON string.
        Do not include markdown.
        Do not include extra text outside the JSON.

        JSON format:
        {{
        "object": "bottle" or "cup",
        "property": "water" or "empty" or "pen",
        "action": "shake and listen" or "tilt and look",
        }}
        """
        
    
    plan = ask_llm(prompt, num_predict=num_predict)
    return plan
    

In [43]:
# COMMAND_LIST = [
#     'fetch me an empty bottle',
#     'fetch me a bottle with water',
#     'give me an empty cup',
#     'give me a cup with pen']

# for c in COMMAND_LIST: 
#     plan = parse_command_llm_with_reasons(c)
#     print(plan)

In [44]:
# for c in COMMAND_LIST: 
#     # plan = parse_resoning_llm(parse_command_llm(c, print_response=True))
#     plan = parse_command_llm(c, num_predict=40)
#     print(plan)

In [45]:
# json.loads(plan)

# Turtlebot

In [ ]:
import math
import rclpy
from rclpy.action import ActionClient

from nav2_msgs.action import NavigateToPose
from geometry_msgs.msg import PoseStamped
from geometry_msgs.msg import PoseWithCovarianceStamped


# 🔹 Replace tf_transformations with this
def yaw_to_quaternion(yaw):
    return (
        0.0,
        0.0,
        math.sin(yaw / 2.0),
        math.cos(yaw / 2.0)
    )


class Navigator:
    def __init__(self, node):
        self.node = node
        self.client = ActionClient(node, NavigateToPose, 'navigate_to_pose')

        # recorded goals to go to
        self.goals = {
            "arm_table": [-0.50607, 1.92922, 2.47735],
            "user_table": [0.332364, 0.577014, -1.04634]
        }


    def go_to(self, goal_name):
        if goal_name not in self.goals:
            print(f"❌ Unknown goal: {goal_name}")
            return

        x, y, yaw = self.goals[goal_name]

        print(f"🚗 Navigating to {goal_name} → ({x:.2f}, {y:.2f}, yaw={yaw:.2f})")

        goal_msg = NavigateToPose.Goal()
        goal_msg.pose = self._create_pose(x, y, yaw)

        # 🔹 wait for Nav2
        if not self.client.wait_for_server(timeout_sec=5.0):
            print("❌ Nav2 action server not available!")
            return

        send_future = self.client.send_goal_async(goal_msg)
        rclpy.spin_until_future_complete(self.node, send_future)

        goal_handle = send_future.result()

        if not goal_handle or not goal_handle.accepted:
            print("❌ Goal rejected")
            return

        print("⏳ Moving...")

        result_future = goal_handle.get_result_async()
        rclpy.spin_until_future_complete(self.node, result_future)

        print("✅ Arrived at destination!")

    def _create_pose(self, x, y, yaw):
        pose = PoseStamped()
        pose.header.frame_id = "map"
        pose.header.stamp = self.node.get_clock().now().to_msg()

        pose.pose.position.x = x
        pose.pose.position.y = y

        qx, qy, qz, qw = yaw_to_quaternion(yaw)

        pose.pose.orientation.x = qx
        pose.pose.orientation.y = qy
        pose.pose.orientation.z = qz
        pose.pose.orientation.w = qw

        return pose

In [ ]:
from turtlebot_nav.navigator import Navigator
navigator = Navigator(node)

In [ ]:
def call_turtlebot():
    print("🤖 Calling Turtlebot for delivery...")

    # 🔹 move robot to the table with kinova arm
    navigator.go_to("arm_table")

    print("Turtlebot arrived, waiting for drop-off")

def send_turtlebot_back():
    print("📦 Delivering object to user...")

    navigator.go_to("user_table")

    print("🎉 Delivery complete!")

In [ ]:
call_turtlebot()

In [ ]:
send_turtlebot_back()

# !!! DEMO!!!

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output
from my_project.visualization  import show_failure, show_success, status_banner
import time

OBJECTS = {
    "bottle": ["bottle1", "bottle2"],
    "cup": ["cup1", "cup2"]
}

COMMAND_LIST = [
    'fetch me an empty bottle',
    'fetch me a bottle with water',
    'give me an empty cup',
    'give me a cup with pen']

dropdown = widgets.Dropdown(
    options=COMMAND_LIST,
    value=None,
    description='Command:',
    layout=widgets.Layout(width='300px')
)


In [55]:
output = widgets.Output()

def on_change(change):
    global PLAN
    if change['type'] == 'change' and change['name'] == 'value':
        cmd = change['new']
        
        if cmd is None:
            return
        
        with output:
            clear_output()
            
            print(f"🗣 Selected command: {cmd}")
            
            time.sleep(1)
            
            plan = parse_command_llm(cmd)   # or parse_command_llm
            PLAN = json.loads(plan)
            print(f"Plan: {PLAN}\n")
            
# dropdown.unobserve_all()
dropdown.observe(on_change)

display(dropdown, output)

Dropdown(description='Command:', layout=Layout(width='300px'), options=('fetch me an empty bottle', 'fetch me …

Output()

In [58]:
trial_name = "find_cup"
# start_video_recording()
start_laptop_record(trial_name)

💻 Laptop mic recording → demo/audio/laptop_find_cup_20260508_173732.wav


'demo/audio/laptop_find_cup_20260508_173732.wav'

Recording WAVE 'demo/audio/laptop_find_cup_20260508_173732.wav' : Signed 16 bit Little Endian, Rate 16000 Hz, Stereo


In [60]:
print(f"Looking for {PLAN['object']} - {PLAN['property']}...")
print(f"Assuming {PLAN['object']} has been detected!")
print(f"Start interactive behavior: {PLAN['action']}")

open_gripper()

if "bottle" in PLAN['object']:   # 1. target object
    want_water = (PLAN['property'] == "water")  # 2. target property
    move_to_pose_robust(grey_bottle_grasp_pose)  # 3. move to pre-defined pos
    close_gripper() 
    if 'shake' in PLAN['action']:   # 4. target action
         
        with_water = shake_bottle(   # 5. interact
            test_num=f"shake_{PLAN['object']}_{PLAN['property']}",
            audio_save_path="demo/audio"
        )  
        move_by_plan(SHAKE_TO_BOTTLE_TRAJ)
        display(status_banner)

        if with_water == want_water:
            print("🎯 Found target object, get ready to drop off")
            show_success()
            # move_to_pose_robust(drop_off_pose)
            # print(f"calling turtlebot...")
            # call_turtlebot()
        else:
            print("➡️ Not correct, try next bottle")
            show_failure()
            # move_by_plan(SHAKE_TO_BOTTLE_TRAJ)
            open_gripper()
    else:
        print(f"action {PLAN['action']} is not suitable for {PLAN['object']}!")
elif "cup" in PLAN['object']:
    want_pen = (PLAN['property'] == "pen")
    move_to_pose_robust(cup_pen_grasp_pose)
    close_gripper()
    if "tilt" in PLAN['action']:
        with_pen = tilt_cup(
            test_num=f"shake_{PLAN['object']}_{PLAN['property']}",
            image_path="demo/image"
        )

        move_by_plan(TILT_TO_CUP_TRAJ)
        display(status_banner)
        
        if with_pen ==  want_pen:
            print("🎯 Found target cup, get ready to drop off")
            show_success()
            # move_to_pose_robust(drop_off_pose)
            # print(f"calling turtlebot...")
            # call_turtlebot()
        else:
            print("➡️ Not correct, try next cup")
            show_failure()
            # move_by_plan(TILT_TO_CUP_TRAJ)
            open_gripper()
else:
    print(f"Object {PLAN['object']} is not eligible!")

Looking for bottle - empty...
Assuming bottle has been detected!
Start interactive behavior: shake and listen
✅ Armd reached target pose
=== Shake Bottle Behavior ===
Arm is in starting position. Executing saved trajectory...
✅ Trajectory completed
start recording audio...
🎤 Start recording → demo/audio/trial_testshake_bottle_empty_20260508_173822.wav


Recording WAVE 'demo/audio/trial_testshake_bottle_empty_20260508_173822.wav' : Signed 16 bit Little Endian, Rate 16000 Hz, Mono


Execute trial 1
✅ Success
⏹ Stop recording


Aborted by signal Terminated...


[Audio] Recorded file: /home/bee-humble/workspace/class_project_ws/src/demo/audio/trial_testshake_bottle_empty_20260508_173822.wav
❌ Empty bottle
Execute trial 1
✅ Success
Arm is in starting position. Executing saved trajectory...
✅ Trajectory completed


HTML(value='\n        <div style=\'text-align:center;\n                    font-weight:900;\n                 …

🎯 Found target object, get ready to drop off


In [61]:
# stop_video_recording(name=trial_name, save_path="demo/video", fps=15)
stop_laptop_record()

⏹ Stop laptop recording


Aborted by signal Terminated...
arecord: pcm_read:2221: read error: Interrupted system call


'demo/audio/laptop_find_cup_20260508_173732.wav'

In [ ]:
# start_laptop_record("test22222")
# time.sleep(0.5)
# stop_laptop_record()

In [56]:
open_gripper()

In [57]:
move_to_pose_robust(cup_pen_pre_grasp_pose)
move_to_pose_robust(grey_bottle_pre_grasp_pose)

✅ Armd reached target pose
✅ Armd reached target pose


True

## Shake

open_gripper()

In [ ]:
open_gripper()
move_to_pose_robust(grey_bottle_grasp_pose)

In [ ]:
move_to_pose_robust(above_mircophone_pose)

In [ ]:
close_gripper()
with_water = shake_bottle(
    test_num=0,
    audio_save_path="demo/audio"
)

display(status_banner)
want_water = (PLAN['property'] == "water")

if with_water == want_water:
    print("🎯 Found target object, calling turtlebot...")
    show_success()
    call_turtlebot()
else:
    print("➡️ Not correct, try next bottle")
    show_failure()
    move_by_plan(SHAKE_TO_BOTTLE_TRAJ)
    open_gripper()

In [ ]:
move_by_plan(moveit_plan_pkl_path="demo/config/trajectory_shake2bottle.pkl")


## tilt cup

In [ ]:
open_gripper()

In [ ]:
output = widgets.Output()
dropdown.observe(on_change)
display(dropdown, output)

In [ ]:
open_gripper()

In [ ]:
close_gripper()

In [ ]:
move_to_pose_robust(cup_pen_pre_grasp_pose)

In [ ]:
move_to_pose_robust(cup_pen_grasp_pose)

In [ ]:
close_gripper()

In [ ]:
want_pen

In [ ]:
has_pen = tilt_cup(
    test_num=0,
    image_path="demo/image"
)

want_pen = (PLAN['property'] == "pen")
display(status_banner)

if has_pen == want_pen:
    show_success()

    print("🚗 Calling Turtlebot...")
    call_turtlebot()

else:
    show_failure()
    print("➡️ Not correct, try next cup")
    move_to_pose_robust(cup_pen_grasp_pose)
    open_gripper()

## delivery

In [ ]:
move_to_pose_robust(drop_off_pose)

In [ ]:
open_gripper()

In [ ]:

send_turtlebot_back()

# shutdown

In [ ]:
node.destroy_node()

rclpy.shutdown()

print("Shutdown complete")